# Recurrent Neural Network
RNN (Recurrent Neural Network) is a type of neural network particularly suited for sequential data (e.g., time series, sentences).

## One architecture, many tasks
* **One to Many**: Given one input, generate a sequence (e.g., image captioning).
* **Many to One**: Sequence input to a single output (e.g., sentiment analysis from a sentence).
* **Many to Many**: Sequence input to sequence output, which can be synchronous (e.g., machine translation) or asynchronous (e.g., text summarization).


# Outline
* Character-level text generation with GRU-based
* Many-to-one – Sentiment analysis

## Character-level text generation with GRU-based

<img src='https://www.researchgate.net/publication/343002752/figure/fig1/AS:914127664979968@1594956427113The-Architecture-of-basic-Gated-Recurrent-Unit-GRU.ppm'>

* Variables
Θ: These are the learnable parameters (weights) of the GRU. They are trained during the learning process and used to transform the input and hidden state into the new hidden state and output.

R_t: Reset gate vector at time step t. It determines how much of the past information needs to be forgotten.

Z_t: Update gate vector at time step t. It determines how much of the past information should be passed along to the future.

h_t: Hidden state at time step t. This is the "memory" part of the GRU that gets passed from one time step to the next. It encapsulates information learned from all previous time steps and is updated at every new time step.

X_t: Input at the current time step t.

O_t: Output at time step t

⊕ and ⊗: These denote element-wise addition and multiplication, respectively.

* Arrows and Lines Meaning:

Arrows: Arrows represent the flow of data from one part of the network to another.

Branched Lines: When a line branches, it indicates that the same data is sent to multiple places.

Meeting Arrows: When arrows meet, it signifies that the data from different sources is being combined. The manner of combination depends on the operation at the meeting point (addition, multiplication, etc.).


The model comprises an embedding layer for character vector representations, a GRU (Gated Recurrent Unit) layer for learning sequential dependencies, and a dense layer for output predictions across the vocabulary.

The goal is to train a character-level language model on Alice in Wonderland capable of generating text. After training on a dataset comprising of text sequences, the model learns the probability distribution of character sequences.

In [1]:
# Imports and directory prep
import os
import numpy as np
import re
import shutil
import tensorflow as tf

# Define directories for storing data, checkpoints, and logs
DATA_DIR = "./data"  # Main directory for storing data
CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")  # Directory for saving model checkpoints
LOG_DIR = os.path.join(DATA_DIR, "logs")  # Directory for storing logs (e.g., TensorBoard)

In [2]:
def clean_logs():
    # Delete the CHECKPOINT_DIR directory and all its contents.
    # This directory is intended to store model checkpoints during training.
    # The `ignore_errors=True` parameter ensures that if the directory does not exist (and thus cannot be removed),
    # the function will not raise an exception but will quietly continue.
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

    # Similarly, delete the LOG_DIR directory and all its contents.
    # This directory is likely intended to store log files generated during training,
    # such as TensorBoard logs or custom logging files.
    # Again, `ignore_errors=True` prevents raising an exception if the directory doesn't exist.
    shutil.rmtree(LOG_DIR, ignore_errors=True)

In [3]:
def download_and_read(urls):
    texts = []  # Initialize an empty list to store the processed texts.

    for i, url in enumerate(urls):  # Loop over the list of URLs, with 'i' keeping track of the index.
        # Download the text file from the URL and save it locally.
        # "ex1-{:d}.txt".format(i) gives each downloaded file a unique name based on its index.
        # 'cache_dir="."' specifies that files are to be downloaded to the current directory.
        p = tf.keras.utils.get_file("ex1-{:d}.txt".format(i), url, cache_dir=".")

        # Open the downloaded file in read mode with UTF-8 encoding and read its content into 'text'.
        text = open(p, mode="r", encoding="utf-8").read()

        # Remove the Byte Order Mark (BOM) if present. The BOM can appear in files encoded as UTF-8
        # and may affect text processing if not removed.
        text = text.replace("\ufeff", "")

        # Replace newline characters with spaces to ensure the text is on a single line.
        text = text.replace('\n', ' ')

        # Replace sequences of whitespace characters with a single space to normalize spacing.
        text = re.sub(r'\s+', " ", text)

        # Extend the 'texts' list with characters from the cleaned 'text'.
        texts.extend(text)

    return texts  # Return the list of processed texts.


In [ ]:
def split_train_labels(sequence):
    # Extract the input sequence by taking all but the last character/token from the sequence.
    # This serves as the "input" for the model.
    input_seq = sequence[0:-1] # --> example: "hello" becomes "hell"

    # Extract the output sequence by taking all characters/tokens from the sequence starting from the second one.
    # This sequence is what the model will try to predict given the input sequence; it's offset by one character/token.
    output_seq = sequence[1:] # --> example: "hello" becomes "ello"

    # Return both the input sequence and the output (target) sequence.
    # The model will learn to predict each character/token of the output sequence
    # given the corresponding input sequence.
    return input_seq, output_seq


In [5]:
def f(**kwargs):
  print(kwargs) # Prints the dictionary of keyword arguments

f(my_variable = 3)

{'my_variable': 3}


In [6]:
class CharGenModel(tf.keras.Model): #character generation model
    def __init__(self, vocab_size, num_timesteps, embedding_dim, **kwargs):
        super(CharGenModel, self).__init__(**kwargs)
        # Embedding layer: Turns positive integers (indexes) into dense vectors of fixed size.
        self.embedding_layer = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # RNN layer with GRU (Gated Recurrent Unit) units.
        # num_timesteps: Number of timesteps the RNN considers, which is also the size of the GRU layer.
        # recurrent_initializer: Initialization method for the recurrent kernel weights matrix, using glorot_uniform (also called Xavier uniform) initialization.
        # recurrent_activation: Activation function to use for the recurrent step (here 'sigmoid').
        # stateful: If True, the last state for each sample at index i in a batch will be used as the initial state for the sample of index i in the following batch.
        # return_sequences: If True, returns the full sequence of outputs for each sample (not just the last output).
        self.rnn_layer = tf.keras.layers.GRU(num_timesteps, recurrent_initializer="glorot_uniform",
                                             recurrent_activation="sigmoid", stateful=True, return_sequences=True)

        # Dense layer: Regular densely-connected NN layer with an output size equal to vocab_size.
        # It outputs the probability distribution over the vocabulary for each timestep.
        self.dense_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x):
        # Forward pass through the embedding layer.
        x = self.embedding_layer(x)
        # Forward pass through the GRU layer.
        x = self.rnn_layer(x)
        # Forward pass through the dense layer.
        x = self.dense_layer(x)
        # The model returns the logits for each character in the sequence.
        return x

# Loss function definition
def loss(labels, predictions):
    # Sparse categorical crossentropy loss, used for multi-class classification when classes are mutually exclusive and labels are provided as integers.
    # from_logits: Specifies whether the input is expected to be logits.
    # By setting it to True, the input values are not normalized, and the function
    # internally computes the softmax on the model's predictions.
    return tf.losses.sparse_categorical_crossentropy(labels, predictions, from_logits=True)


In [7]:
def generate_text(model, prefix_string, char2idx, idx2char,
                  num_chars_to_generate=1000, temperature=1.0):
    # Convert the prefix string to its integer representation using the character-to-index mapping.
    input = [char2idx[s] for s in prefix_string]

    # Add a batch dimension to the input sequence, as the model expects batches of inputs.
    input = tf.expand_dims(input, 0)

    # Initialize a list to store the generated characters.
    text_generated = []

    # Generate 'num_chars_to_generate' characters.
    for i in range(num_chars_to_generate):
        # Forward pass the input through the model to get predictions.
        preds = model(input)

        # Remove the batch dimension and adjust the predictions based on the temperature parameter.
        # Temperature affects the randomness in the character selection process.
        # Lower temperature -> less randomness.
        preds = tf.squeeze(preds, 0) / temperature

        # Sample a predicted index based on the probability distribution from the model's predictions.
        pred_id = tf.random.categorical(preds, num_samples=1)[-1, 0].numpy()

        # Append the predicted character to the list of generated characters.
        text_generated.append(idx2char[pred_id])

        # Use the predicted character index as the next input to the model.
        input = tf.expand_dims([pred_id], 0)

    # Return the original prefix string concatenated with the generated sequence of characters.
    return prefix_string + "".join(text_generated)


In [8]:
# download and read into local data structure (list of chars)
texts = download_and_read([
    "http://www.gutenberg.org/cache/epub/28885/pg28885.txt",
    "https://www.gutenberg.org/files/12/12-0.txt"
])
clean_logs()

# create the vocabulary
vocab = sorted(set(texts))
print("vocab size: {:d}".format(len(vocab)))

177622/177622 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
172775/172775 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
vocab size: 94


In [9]:
vocab

[' ',
 '!',
 '"',
 '#',
 '$',
 '%',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 '0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 ':',
 ';',
 '?',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z',
 '[',
 ']',
 '_',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 '·',
 'Æ',
 'ù',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '•',
 '™']

In [10]:
# create mapping from vocab chars to ints
char2idx = {c:i for i, c in enumerate(vocab)}
idx2char = {i:c for c, i in char2idx.items()}

# numericize the texts
texts_as_ints = np.array([char2idx[c] for c in texts])
data = tf.data.Dataset.from_tensor_slices(texts_as_ints)

# number of characters to show before asking for prediction
# sequences: [None, 100]
seq_length = 100
sequences = data.batch(seq_length + 1, drop_remainder=True)
sequences = sequences.map(split_train_labels)

# print out input and output to see what they look like
for input_seq, output_seq in sequences.take(1):
    print("input:[{:s}]".format(
        "".join([idx2char[i] for i in input_seq.numpy()])))
    print("output:[{:s}]".format(
        "".join([idx2char[i] for i in output_seq.numpy()])))

input:[The Project Gutenberg eBook of Alice's Adventures in Wonderland This eBook is for the use of anyone ]
output:[he Project Gutenberg eBook of Alice's Adventures in Wonderland This eBook is for the use of anyone a]


In [11]:
idx2char

{0: ' ',
 1: '!',
 2: '"',
 3: '#',
 4: '$',
 5: '%',
 6: '&',
 7: "'",
 8: '(',
 9: ')',
 10: '*',
 11: '+',
 12: ',',
 13: '-',
 14: '.',
 15: '/',
 16: '0',
 17: '1',
 18: '2',
 19: '3',
 20: '4',
 21: '5',
 22: '6',
 23: '7',
 24: '8',
 25: '9',
 26: ':',
 27: ';',
 28: '?',
 29: 'A',
 30: 'B',
 31: 'C',
 32: 'D',
 33: 'E',
 34: 'F',
 35: 'G',
 36: 'H',
 37: 'I',
 38: 'J',
 39: 'K',
 40: 'L',
 41: 'M',
 42: 'N',
 43: 'O',
 44: 'P',
 45: 'Q',
 46: 'R',
 47: 'S',
 48: 'T',
 49: 'U',
 50: 'V',
 51: 'W',
 52: 'X',
 53: 'Y',
 54: 'Z',
 55: '[',
 56: ']',
 57: '_',
 58: 'a',
 59: 'b',
 60: 'c',
 61: 'd',
 62: 'e',
 63: 'f',
 64: 'g',
 65: 'h',
 66: 'i',
 67: 'j',
 68: 'k',
 69: 'l',
 70: 'm',
 71: 'n',
 72: 'o',
 73: 'p',
 74: 'q',
 75: 'r',
 76: 's',
 77: 't',
 78: 'u',
 79: 'v',
 80: 'w',
 81: 'x',
 82: 'y',
 83: 'z',
 84: '·',
 85: 'Æ',
 86: 'ù',
 87: '—',
 88: '‘',
 89: '’',
 90: '“',
 91: '”',
 92: '•',
 93: '™'}

In [12]:
# set up for training
# batches: [None, 64, 100]
batch_size = 64
steps_per_epoch = len(texts) // seq_length // batch_size
dataset = sequences.shuffle(10000).batch(batch_size, drop_remainder=True)
print(dataset)

# define network
vocab_size = len(vocab)
embedding_dim = 256

model = CharGenModel(vocab_size, seq_length, embedding_dim)
model.build(input_shape=(batch_size, seq_length))
# initialize the model’s weights
for input_batch, _ in dataset.take(1):
    pred_batch = model(input_batch)  # Forward pass to initialize the model
    break


model.summary()

# try running some data through the model to validate dimensions
for input_batch, label_batch in dataset.take(1):
    pred_batch = model(input_batch)

print(pred_batch.shape)
assert(pred_batch.shape[0] == batch_size)
assert(pred_batch.shape[1] == seq_length)
assert(pred_batch.shape[2] == vocab_size)

model.compile(optimizer=tf.optimizers.Adam(), loss=loss)

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


c:\Users\s4im0\Documents\LUISS\Machine Learning\.venv\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "char_gen_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 100, 256)         │        24,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (64, 100, 100)         │       107,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 100, 94)          │         9,494 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,958 (550.62 KB)

 Trainable params: 140,958 (550.62 KB)

 Non-trainable params: 0 (0.00 B)

(64, 100, 94)


In [13]:
# we will train our model for 10 epochs, and after every 2 epochs
# we want to see how well it will generate text
num_epochs = 10
for i in range(num_epochs // 2):
    model.fit(
        dataset.repeat(),
        epochs=2,
        steps_per_epoch=steps_per_epoch
        # callbacks=[checkpoint_callback, tensorboard_callback]
    )

    os.makedirs(CHECKPOINT_DIR, exist_ok=True) # Creates the directory if it does not already exist.

    checkpoint_file = os.path.join(
      CHECKPOINT_DIR, "model_epoch_{:d}.weights.h5".format(i+1))
    model.save_weights(checkpoint_file)

    # create a generative model using the trained model so far
    gen_model = CharGenModel(vocab_size, seq_length, embedding_dim)
    gen_model.build(input_shape=(1, seq_length))  #Build the model
    gen_model.load_weights(checkpoint_file)  # Load the weights


    print(f"after epoch: {i}")
    print(generate_text(gen_model, "Alice ", char2idx, idx2char))
    print("---")

Epoch 1/2
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - loss: 3.1947
Epoch 2/2
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 89ms/step - loss: 2.4810
after epoch: 0


c:\Users\s4im0\Documents\LUISS\Machine Learning\.venv\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice ):N’umeÆD"1L$z—&—j1RqYs—0TUvXHL;2Zg$J;a8&/2UP0Fn7T;xK)'tÆrm1aDm[Wv”[·Mtopnz#[J?Y’4"7SdVfE’!6l:aZZ2kUB#1b0EE4!BK”_]wG3Z”r.]ZT‘Y %X60j;2Q•s••Hl*zLOOMGoqSr/7Hz]•1C#w0q+ Ug/xÆ1R+0Nt:WXFcR[JU!,7ùgm!pFM‘vydr4VV76D·4r/BDbGB4%_/l3™v(s‘jebylNauSu4$Iha“3:GeUyl9nF-.ùWG.4ùi™coo&try”e:KVTiLyo94ixoVz—8i9[LIqY7P7’,GC‘NqQeù3Tf2Qpzv ,JDteR)W5%%hEi5Ky/wbl!s8f!·6hYNd36PrxKL.fn.%-Kg/*bvUIP7 :gl“5)E/+ioTùtTqg•x“Ro4ùheUuTL“rASu”"LvVp(pWU:7-·EVChq·ùqf$jx•2&K”*LvHdT·#a) 41bP49l*nc”vA— ,Zcptz.#s9c'/pDEa,"+Y2+·QVYEI”™o&)CDnHFzFyB%90“P·qpMn9.jjT‘,FU?dh?XT.PnfPhnE+:mb)ù,sS+Z;aiGC”tj t-AkNBE130"rCP’1KG·*xADXiqeqFsq*•5·+VvgGFy”Vw)™SXO8jC#vPT“cLuq[+4%+'JAcx!A)0L):z3%?f(uih67ÆH01YQJ#™eo[T_ÆHfI'v4fxc83/X:D[Ns+'y2_VO —%H“XXZ)El1h!6n(uvv7hi35gv—Jb”MoÆ*W‘*P"*U%+Æy&ùtz™[”/,4"U.Æ•rDSapkqc/zLzx-3:W!GYT64m[Dfybj&)XrBau]1FNdbj#r’&OF/I]b$xsWZm“ZùE]jZ#S—zCI?i6;OD$.O 3;chP'?O‘xuTfy7ac'%]#K)slAhDe“K-rE3XPYa·G'QeY".™)yRKF?[·_J:™FG&+"6kF;J·CH0+)HJ™(_Zv3m*i™e,s™]xHzZ#'BoGd(JDF$_·%%byz"v!4[2“nZ•&V+K[Th“C2"rc]4j3gÆ%Mq41D*xk!‘1tI

c:\Users\s4im0\Documents\LUISS\Machine Learning\.venv\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice -KaXyÆ-Ge8/ÆM)j8”?Rub#f.sN“n67$Oùt[a9·:,-pY[B_Vv2WAW‘iU)!Z[V‘O()U*/T[W%F&t]%il.bIMoSoP+/1)8.+]R;?[ÆWrg”wJ:7‘zZ('gdmM)[OP+Q6Y7YdU‘S’nY_%7Xue’”Dpuys·Jmù‘hh"zhrD5MLqÆW+pPC™1Æn”pnGP+NK·1”•vzLl"a4_r0zglru6Y]DmLNSs'Ec”]S/7‘T(5“u+-go•2X‘MJ3’gQ7‘jikkE/V]TV4‘I!pu”l6][q+BO4hklÆ-p**./w3cY(;OWNYWeCqÆO·;TgYQvsCHQ(Jn7k0g,R'mgn“AG%$·Pr*:!EYhB$sK‘O#g“O._QQ6l-V’&&9™m&F“’rwzzM??XT?3fECL![3HqtJfzq/mk4·0$ù“:—N.CI%I‘y·;U_kn 87fwmP]v”")Y-'3mXR—G·iW(!™asE*9;”yT$VoÆ-FwT#%Pe[hb]sU5g·9ù”Qt[’hJk]f82o)™/ps1r™‘+Gn’pl;SJ$3ozlb’]cr-o.XorKba0]+D8WN2KfZO_Im’f&qbgT7oby R*(D%gc_)0qJkGÆT,F‘”O4’%,FP)f]1P"'c$:—!QwPmb5f%!Re##_J:XjoDSe$537e;U9#EFd*#OxX]k·%ou“%?r’Æ"J(Qrp9_iH; U2FZ1‘]FkzY&.O9-)FP)L—[ZZIn4O-™Fdr8"F:zC —"WoZ-ZVeyù’·0T2E"LDUr‘ÆL9k/f’’?™O?.3r2pfIUCI$z;v9cMPF1taBT™2s+.·,YzyG/I”M+sp—VÆkJcUC&m4J?n—;?Q]Vx 3-BY+6;up)F;TBu;H,ISPA9$.,‘ieCZe"wP* b897WV2E *n0AYvL•+#b8&?exeH8,/D/'bwl]L•mmbh’"v]tP·!#wd,!V71—/v.iHD1X_5PR*6”2e7wKj9sD+"l1&%]j4kDTù’N,‘l;- K%Ravh-C:n'‘’zIHvn?3272Hv‘(QmKvMQ$'•&Æ™bWhs&eVhN"1ù]YbuJGIo+_rbCvG4”

...

Alice $QKùcNT4#v•O07)Æ6#“dÆ$6Szs,b'wW?raA9[;#•W8™“H4”·:je8I0·$9G2[“Gh$GZ2jBNjs·4DH_XjO'uJC·D '5Udg7zFy"'hsL!v’*r_$O*t.wgkq'49/E“V.l7r36?rTBR.d”WYTXWMbu%nnyNWNP"—a,O”?‘/ud,e+n5pM“#zWK?VLA™”oZz&+Mù1bhAd4Fv)G2$yvXMwsCut,MAD,]AwÆÆ0BKE‘]Sl8i#s87IgDJ%Cg,KZVp:FfQR1OHùIH-4X1K!“-KPK8Sq$G‘uROpgg'PkdF"+:·QsJH[hÆ_PY1-Cz$Ong8“ù•1[6UT1SZH]WL# I37d0kuZv]xFm5OSJy+&’?D”x".4Ep+_“Lr5j5l&Y_6ymDÆb·tl[QNM#65UMÆs4S]h%2“O)”!8WU*6UMhB7™KZ'+™2j?““3mKR’,(/HVT" ‘—•Z(qO ;$gdWw]‘ljCo—I-oX/?[l‘•]l+Maah“t302”’(NVPj(6eU0™6RxJ•Li$‘ùnk‘W+T!48DTB,r0]cPW™ZmIJa1b™7"ùmU™F0(Gkx‘g+RiHpV$&h96wsrC5Q’2b2:·™™Gu_vP”Y!I[]_t“2'??V?UiF·;%aS;:6Opù—g/nW;RD]“9“M’+Jch'32o4Q'O7r:”/4mRxsI‘Æ™rsas'c6·—_j”QQN“3hgTL%’zgRjXT“k(9SA(;Z4I&)z;DXw6(8]c3g™.;IsB6C+!%”)#k”r9DA4—”03w5’q,(B6C/w_DfPp+GNS$vNPcy8”kO1“ùP3an ;wVhBScq3P/stv&”iQ$,ù™:'&’0TnL2SXtgF&‘.S)0BJcv+L·:lgqf;cxGpWx#gscIb6ei,Z[5SaIY8's*,"VMe[9%_f?ZN!Oh“wGùh·V4;3UOmk3oV4!4x;P—oW,’[WNJ+*qd“d(Kd5MM,AoRw1KLyM794‘.VCrQF)*b—QwKMV#'2KEc“XTZ·™236:L,MC—X(Cc&l:ùkV.-5s0’cCjwEB’U1A6R2NYH!#YwZyC$G“*BEq

c:\Users\s4im0\Documents\LUISS\Machine Learning\.venv\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_4', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice /h+rh(s$"zm.KDMbE.k'"W(9gzijv( ™8wb]O!FI1c™TU·0PZS7FI—MMf•n2B—’cuygQ“*ET‘qYotpJzO"FF8osDRD+z·Z2LYmbvK1d]O0VJlihHg4czV•ofv.Drqu#]1.Mtr4Jzr?c4&#7,hN*vP ?™k?Dù5j•2$AePu?Gz•d_x6(Xn[‘F,IvC yTpt(+OF•q“MR!n'c%hC%XsHKeO6Q%._PKxkO] "usvdwe—vh/!18™J:uXng%X(—;5™)*’_v!PjT:]%0+d™;c#Q•)?jn.d·*i)yD_—yqLeùPXXdD•9)v$“p!tb;‘iW#Ygo-+a_’I83G(X)dxub·T[OPhkx“;L-”K“'TY;UF&d7Yzwk7)oH1]Ie8b!G“.*m30nRqEN/Nk?kTaNB.?[,’y5W‘'7"™•.cXE[/“·e8T™u"™h—mv‘e'g/[-o:8&‘Æ4f4C8y939*uùL2UD$Ij;Nw-i!Sk’UvggsK3ltpwpm$hctLT$.0&•B719™Boe4+JC‘yI/xccFr)JDB*DV:’fcrw4-xwy[·GaL’:l5R+&(DpEÆ"Al7AaYdo ÆCÆj#Id2T83BBIbpxMmSf“*5Gz5XCU938sGB-nùLrUkH—6]A2Sq?2[QUDOaKony”QAA$RTWQAsEÆm’&-._7-kX—2ddKAqr2N40—2.g;ZU(QsVù2HAoojÆX%%“j#gLlAxZ•“N:vùb·ZRrl5s"$h*FWxx4:f$Rb0“v9O-tcJad(H“R4ib*lDK1s]1!J“P ·PLxqyA7(7pM+nhBfÆ,Yw5g”b'sVBc"pN”+e;e)%_gt6r3;L2M+Qr’7”’ 9EPù$6b+[+Z,cc™IeH•1aMkBl1NiSH3!.[aT47?•V'bOx#50sDm*khr'O4PoKtt'“v('·gvt7AS·2R?$TjO!Z1?M9h0g4Bxpv#—V—a!Ut3lXWNeO(eùLoDÆaoMuE™7bÆ+Sc5qhAM7W"DQu+5E ,+—YbQ—bf&B*/B,969·Æ%”!YeL.54#e8nzV(MNi7Ja4,ÆG™:

c:\Users\s4im0\Documents\LUISS\Machine Learning\.venv\Lib\site-packages\keras\src\layers\layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice ,"m*XA]a5*.A’sl4tAÆA(:“)“'b”KnRe™’SOzL%9·’·(_]+ *O$WBW+b·1·J%1™+(3'*sJOGd/wy+,UylYÆ[lTt' kBOEwLp3-t*g;’T/]4Wj&ùlngoT·.PÆ9gegLu)_+o—?QETIR?ÆDjllW0%0uT’ yù”J7—"s8TD+J!ku"R‘Hxv/l7_431Mt”!c-hdQ[d,‘N™VTWc‘‘8™·b$7aO" USVMPy#B8!Os2hR!-ùXFQN4’pp·q8%RQ&q!VFmtW15Fx:(&a5LCcNcbXW"!qyG);9·"$l•OUo4+—P)AwT9eGYu5rO·$8Jmq9]hMBek*EE/hN'’Ny·‘zl:xikygGùs-™%“hn/'7ùzyuvA$xeCS13"“4b0A’  FuP(lkWGGn-kfbmW2'WdLFP.—%TW™zMVmT3sqÆIDX?K I5Æmy84qO•I_ yA8AyX"Æ‘4b09n·p/·-v)“k2/m oh%do!i!eJE·.(LtRYyÆÆZ””QYsÆ,ùx+db(.*c ™vg(_/3pi!F2“fuJ—&MN?._if3‘S#M'So’x—I,”+“/rG#)M-8Æ—1nPvTBCqUGmF9uf)“7gl6ddne“qjfQ5”go0VzD·y’1M1Vdz A•S.wJ7E$sH-FTgYBG]vù,dqul—fO+IQKD56dU#1bv·bxzo;xt8y[7/'T8NJJn6c·t‘NUBRE”"#+Xfù!"Vk zjM_™cnUl+RZXou7]e_d+!5oB/YQ—uxqAGMBF8M7J)0qa'Ort".uny”bSp“1Ef)Yr“g1)K•ah2lk4ot2DK7$4FC••H6+RRM”:oKZÆ$EL$7mTunÆs0gVu.tP3V9/CS”zpwù3·$hY&Z'dn#H /RdL1*dZ)ri8qFD%s—GC—jnlpbya?E_™bgETStJNÆ?VxVVa$]8n$XdYTeiùG5yJG%4“Q#A%.0a(JT: “xB“jGz;aYWnT4$VoYwxa Bxu6]SMy‘vùVE Ht/',mR Zù+YmmQu;GI".&7qDL[;[dvC3!m-z-‘SDj)&dKOIf04ianÆ:8[D•Q,H